In [ ]:
# Imports and setup
import json
import os
import sys
from typing import List, Optional, Dict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm

sys.path.append('..')
sys.path.append('../..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    plot_cost_vs_safety_curves,
    get_roc_curve,
    interpolate_tpr,
    prepare_monitoring_data,
    merge_monitoring_with_combined_results,
    plot_training_progress,
    plot_all_suspiciousness_metrics,
    is_in_training_data,
    filter_and_prepare_data,
    prepare_data_for_training_progress,
)
from utils.analysis_utils import extract_verdict
from utils.path_utils import resolve_path


In [ ]:
# Constants and configuration
MAIN_AGENT_MODELS = ['claude-3-5-sonnet-20241022']

# Step mapping for fine-tuned models (second series)
default_step_mapping = {
    "gpt-4o-2024-08-06": 0,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1ENZ3:ckpt-step-140": 140,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1Fwer:ckpt-step-210": 210,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1FbkP": 280,
}
FT_MONITOR_MODELS = list(default_step_mapping.keys())
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022'] + FT_MONITOR_MODELS
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]

# Tasks to exclude from analysis
EXCLUDED_TASKS = ['election_tampering', 'art_collectors', 'blog_post', 'spam_filter_update']

# Cache for combined_results
_combined_results_cache = {}
NUM_MONITORS = 3


In [ ]:
# Helper functions for data loading and preparation
def load_combined_results(include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and cache combined_results."""
    cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
    if cache_key in _combined_results_cache:
        print(f"Using cached combined_results for {cache_key}")
        return _combined_results_cache[cache_key]
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    _combined_results_cache[cache_key] = combined_results
    return combined_results

def prepare_system_prompt_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and prepare the initial datasets."""
    # Use the generalized function from plotting_utils
    system_prompt_df = prepare_monitoring_data(
        base_dir=base_dir,
        monitor_models=FT_MONITOR_MODELS,  # Filter to only include fine-tuned models
        num_monitors=NUM_MONITORS,  # Process all monitor responses
        compute_aggregates=True,  # Compute max, mean, etc.
        tag_word='verdict'
    )
    
    return system_prompt_df

def merge_and_compute_features_for_system_prompt(combined_results: pd.DataFrame, 
                             system_prompt_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    return merge_monitoring_with_combined_results(
        monitoring_df=system_prompt_df,
        combined_results=combined_results,
        how='left',  # Use left join as in the original
        compute_convo_length=True
    )


In [ ]:
# Load the base data
combined_results = load_combined_results(include_failed_attempts=False, include_failed_main_task=False)

# Load system prompt data from different system prompts
system_prompt_df_1 = prepare_system_prompt_data(base_dir='monitor_responses/system_prompt_1', include_failed_attempts=False, include_failed_main_task=False)
system_prompt_df_5 = prepare_system_prompt_data(base_dir='monitor_responses/system_prompt_5', include_failed_attempts=False, include_failed_main_task=False)
system_prompt_df_0_temp = prepare_system_prompt_data(base_dir='monitor_responses/system_prompt_5_temp_0', include_failed_attempts=False, include_failed_main_task=False)

# Add side_task_evaluation column
system_prompt_df_1['side_task_evaluation'] = system_prompt_df_1.transcript_type != 'transcripts_no_secondary_task'
system_prompt_df_5['side_task_evaluation'] = system_prompt_df_5.transcript_type != 'transcripts_no_secondary_task'
system_prompt_df_0_temp['side_task_evaluation'] = system_prompt_df_0_temp.transcript_type != 'transcripts_no_secondary_task'


In [ ]:
# Load training data
training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
training_data = pd.read_csv(training_data_path)

# Identify which examples are in the training data
system_prompt_df_1['in_training_data'] = system_prompt_df_1.apply(
    lambda row: is_in_training_data(row, training_data), 
    axis=1
)
system_prompt_df_5['in_training_data'] = system_prompt_df_5.apply(
    lambda row: is_in_training_data(row, training_data), 
    axis=1
)
system_prompt_df_0_temp['in_training_data'] = system_prompt_df_0_temp.apply(
    lambda row: is_in_training_data(row, training_data), 
    axis=1
)


In [ ]:
# Merge the system prompt dataframes and add combined metrics
both_system_prompts_df = pd.merge(
    system_prompt_df_1,
    system_prompt_df_5,
    on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name', 'in_training_data', 'side_task_evaluation'],
    suffixes=('_sp1', '_sp5'),
    how='inner'
)

# Add computed metrics
both_system_prompts_df['total_max'] = both_system_prompts_df[['max_suspiciousness_sp1', 'max_suspiciousness_sp5']].max(axis=1)
both_system_prompts_df['total_mean'] = both_system_prompts_df[['mean_suspiciousness_sp1', 'mean_suspiciousness_sp5']].mean(axis=1)
both_system_prompts_df['total_min'] = both_system_prompts_df[['min_suspiciousness_sp1', 'min_suspiciousness_sp5']].min(axis=1)

# Filter out excluded tasks
both_system_prompts_df = both_system_prompts_df[~both_system_prompts_df.task_name.isin(training_data.task_name.unique())]

# Add standard deviation metrics
for suffix in ['_sp1', '_sp5']:
    both_system_prompts_df[f'std_dev{suffix}'] = both_system_prompts_df[
        [f'suspiciousness_{i}{suffix}' for i in range(1, NUM_MONITORS + 1)]
    ].std(axis=1)

# Add standard deviation for temp_0 dataframe
system_prompt_df_0_temp['std_dev_sp0'] = system_prompt_df_0_temp[
    [f'suspiciousness_{i}' for i in range(1, NUM_MONITORS + 1)]
].std(axis=1)


In [ ]:
# Compare standard deviation across models
print("Standard deviation of suspiciousness scores by model:")
std_dev_sp1 = both_system_prompts_df.groupby('monitor_agent').agg({'std_dev_sp1': 'mean'})
std_dev_sp5 = both_system_prompts_df.groupby('monitor_agent').agg({'std_dev_sp5': 'mean'})
std_dev_sp0 = system_prompt_df_0_temp.groupby('monitor_agent').agg({'std_dev_sp0': 'mean'})

print("System prompt 1:")
print(std_dev_sp1)
print("\nSystem prompt 5:")
print(std_dev_sp5)
print("\nSystem prompt 5 (temp=0):")
print(std_dev_sp0)


In [ ]:
# Merge the temperature comparison dataframes
both_temp_system_prompts_df = pd.merge(
    system_prompt_df_5,
    system_prompt_df_0_temp,
    on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name', 'in_training_data', 'side_task_evaluation'],
    how='inner',
    suffixes=('_temp_1', '_temp_0')
)

# Filter out excluded tasks
both_temp_system_prompts_df = both_temp_system_prompts_df[~both_temp_system_prompts_df.task_name.isin(training_data.task_name.unique())]

# Display summary statistics for standard deviation
print("Summary statistics for standard deviation (temp=0):")
print(both_temp_system_prompts_df.std_dev_sp0.describe())

In [ ]:
both_temp_system_prompts_df.columns

In [ ]:
DATA_TO_USE = 'no_training_tasks'  # Options: 'all_test', 'no_training_tasks', 'training_tasks_no_train_data', 'training_data_only'
aggregation_type = 'min_suspiciousness_temp_1'

for hint in [True, False]:
    # Use the new utility function to prepare data for plotting
    unstacked_df = prepare_data_for_training_progress(
        df=both_temp_system_prompts_df,
        aggregation_type=aggregation_type,
        step_mapping=default_step_mapping,
        data_to_use=DATA_TO_USE,
        training_data=training_data,
        exclude_tasks=[]
    )
    # Filter based on hint preference
    transcript_filter = 'transcripts_no_hint' if hint else 'transcripts'
    filtered_df = unstacked_df[unstacked_df.transcript_type != transcript_filter]
    
    # Plot ROC curves
    plot_roc_curves(
        both_temp_system_prompts_df, 
        combined_results, 
        give_hint=hint, 
        include_failed_attempts=False, 
        include_failed_main_task=False, 
        monitors=both_temp_system_prompts_df.monitor_agent.unique(), 
        metrics=['min_suspiciousness_temp_1', 'mean_suspiciousness_temp_1', 'max_suspiciousness_temp_1', 'min_suspiciousness_temp_0', 'mean_suspiciousness_temp_0', 'max_suspiciousness_temp_0']
    )

In [ ]:
both_temp_system_prompts_df[both_temp_system_prompts_df.monitor_agent == 'ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1FbkP'].task_name.value_counts()
both_system_prompts_df[both_system_prompts_df.monitor_agent == 'ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1FbkP'].task_name.value_counts()
system_prompt_df_0_temp.task_name.value_counts()

In [ ]:
# Plot ROC curves for different data filters
DATA_TO_USE = 'no_training_tasks'  # Options: 'all_test', 'no_training_tasks', 'training_tasks_no_train_data', 'training_data_only'
aggregation_type = 'min_suspiciousness_sp5'

for hint in [True, False]:
    # Use the new utility function to prepare data for plotting
    unstacked_df = prepare_data_for_training_progress(
        df=both_system_prompts_df,
        aggregation_type=aggregation_type,
        step_mapping=default_step_mapping,
        data_to_use=DATA_TO_USE,
        training_data=training_data,
        exclude_tasks=EXCLUDED_TASKS
    )
    
    # Filter based on hint preference
    transcript_filter = 'transcripts_no_hint' if hint else 'transcripts'
    filtered_df = unstacked_df[unstacked_df.transcript_type != transcript_filter]
    
    # Plot ROC curves
    plot_roc_curves(
        filtered_df, 
        combined_results, 
        give_hint=hint, 
        include_failed_attempts=False, 
        include_failed_main_task=False, 
        monitors=[aggregation_type], 
        metrics=list(default_step_mapping.values())
    )


In [ ]:
# Filter data for analysis based on selected data subset
if DATA_TO_USE == 'all_test':
    interesting_df = filter_and_prepare_data(
        both_system_prompts_df,
        data_to_use='all_test',
        training_data=training_data,
        exclude_tasks=EXCLUDED_TASKS
    )
elif DATA_TO_USE == 'no_training_tasks':
    interesting_df = filter_and_prepare_data(
        both_system_prompts_df,
        data_to_use='no_training_tasks',
        training_data=training_data,
        exclude_tasks=EXCLUDED_TASKS
    )
elif DATA_TO_USE == 'training_tasks_no_train_data':
    interesting_df = filter_and_prepare_data(
        both_system_prompts_df,
        data_to_use='training_tasks_no_train_data',
        training_data=training_data,
        exclude_tasks=EXCLUDED_TASKS
    )
else:
    interesting_df = filter_and_prepare_data(
        both_system_prompts_df,
        data_to_use='training_data_only',
        training_data=training_data,
        exclude_tasks=EXCLUDED_TASKS
    )


In [ ]:
# Plot training progress curves for specific metrics
training_progress_df = plot_training_progress(
    df=interesting_df,
    metric='min_suspiciousness_sp1',
    step_mapping=default_step_mapping,
    model_column='monitor_agent',
    models=FT_MONITOR_MODELS,
    target_fpr=0.03
)


In [ ]:
# Plot all suspiciousness metrics
all_metrics_df = plot_all_suspiciousness_metrics(
    df=interesting_df,
    metrics_list=[
        'max_suspiciousness_sp1', 'max_suspiciousness_sp5', 
        'mean_suspiciousness_sp1', 'mean_suspiciousness_sp5', 
        'min_suspiciousness_sp1', 'min_suspiciousness_sp5'
    ],
    step_mapping=default_step_mapping,
    model_column='monitor_agent',
    models=FT_MONITOR_MODELS,
    target_fpr=0.03
)


In [ ]:
# Compare average suspiciousness scores across models
model_comparison = both_system_prompts_df.groupby('monitor_agent').agg({
    'suspiciousness_1_sp1': 'mean', 
    'suspiciousness_1_sp5': 'mean'
})

print("Average suspiciousness scores by model:")
print(model_comparison)


In [ ]:
# Plot suspiciousness distributions for non-default transcript types
non_default_df = both_system_prompts_df[both_system_prompts_df.transcript_type != 'transcripts_no_hint']
plot_suspiciousness_distributions(
    non_default_df, 
    monitors=FT_MONITOR_MODELS[-1:], 
    metrics=['min_suspiciousness_sp5']
)


In [ ]:
# Find examples with low suspiciousness despite having side tasks
suspicious_examples = interesting_df[
    (interesting_df.side_task_evaluation == 1) & 
    (interesting_df.monitor_agent == FT_MONITOR_MODELS[-1])
].sort_values('total_mean', ascending=True).head(10)

print("Examples with low suspiciousness scores despite having side tasks:")
print(suspicious_examples[['task_name', 'transcript_type', 'total_mean', 'min_suspiciousness_sp1', 'min_suspiciousness_sp5']])
